# Feature Importance & SHAP Analysis

Using the final XGBoost model, we identify which features drive reading and math scores.

Two complementary approaches:
1. **XGBoost `feature_importances_`** — gain-based, fast, gives a global ranking
2. **SHAP TreeExplainer** — model-agnostic explanation with direction (positive/negative effect)

One-hot encoded dummies are grouped back to their original variable for cleaner interpretation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from preprocessing import prepare_data
from models import train_xgboost

df_raw = pd.read_csv('kinder_clean.csv')

(
    X_train, X_test,
    y_read_train, y_read_test,
    y_math_train, y_math_test,
    preprocessor, num_cols, cat_cols
) = prepare_data(df_raw)

pipe_read, pipe_math = train_xgboost(preprocessor, X_train, y_read_train, y_math_train)
print('XGBoost models trained.')

## 1. XGBoost Feature Importances (gain)

In [ ]:
def get_feature_importances(pipe):
    ohe = pipe.named_steps['prep'].transformers_[1][1]
    cat_features = ohe.get_feature_names_out(cat_cols)
    feature_names = np.concatenate([num_cols, cat_features])
    importances = pipe.named_steps['model'].feature_importances_
    return pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)

fi_read = get_feature_importances(pipe_read)
fi_math = get_feature_importances(pipe_math)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, fi, title in zip(axes, [fi_read, fi_math], ['Reading', 'Math']):
    top = fi.head(10)
    ax.barh(top['feature'][::-1], top['importance'][::-1], color='steelblue')
    ax.set_title(f'Top 10 Drivers — {title} (XGBoost)')
    ax.set_xlabel('Feature Importance')
plt.tight_layout()
plt.show()

## 2. SHAP Analysis

`TreeExplainer` is used for efficiency with XGBoost.
OHE dummies are grouped back to their original variable (e.g. all `lunch_*` dummies → `lunch`).

In [ ]:
shap.initjs()

preprocessor_fitted = pipe_read.named_steps['prep']
X_train_enc = preprocessor_fitted.transform(X_train)
raw_feature_names = preprocessor_fitted.get_feature_names_out()
base_feature_names = np.array([n.split('__')[-1] for n in raw_feature_names])

xgb_read  = pipe_read.named_steps['model']
xgb_math  = pipe_math.named_steps['model']

explainer_read = shap.TreeExplainer(xgb_read)
explainer_math = shap.TreeExplainer(xgb_math)

shap_values_read = explainer_read.shap_values(X_train_enc)
shap_values_math = explainer_math.shap_values(X_train_enc)

print('SHAP values computed.')

In [ ]:
# Group OHE dummies back to original variable names
groups = {
    'lunch':           [c for c in base_feature_names if c.startswith('lunch_')],
    'ethnicity':       [c for c in base_feature_names if c.startswith('ethnicity_')],
    't_ethnicity':     [c for c in base_feature_names if c.startswith('t_ethnicity_')],
    'class_type':      [c for c in base_feature_names if c.startswith('class_type_')],
    'birth_quarter':   [c for c in base_feature_names if c.startswith('birth_')],
    'degree':          [c for c in base_feature_names if c.startswith('degree_')],
    'school_location': [c for c in base_feature_names if c in ['school_rural','school_urban','school_suburban']],
}

def group_shap(shap_vals, base_names, groups_dict):
    rows, used = [], set()
    for grp, cols in groups_dict.items():
        idxs = [np.where(base_names == c)[0][0] for c in cols if c in base_names]
        if idxs:
            vals = shap_vals[:, idxs]
            rows.append((grp, vals.mean(), np.abs(vals).mean()))
            used.update(cols)
    for i, name in enumerate(base_names):
        if name not in used:
            rows.append((name, shap_vals[:, i].mean(), np.abs(shap_vals[:, i]).mean()))
    return pd.DataFrame(rows, columns=['feature', 'mean_shap', 'mean_abs_shap']).sort_values('mean_abs_shap', ascending=False)

shap_read = group_shap(shap_values_read, base_feature_names, groups)
shap_math = group_shap(shap_values_math, base_feature_names, groups)

print('Top Grouped SHAP Drivers — READING:')
display(shap_read.head(10))

print('\nTop Grouped SHAP Drivers — MATH:')
display(shap_math.head(10))

In [ ]:
# SHAP beeswarm plots
print('SHAP Summary — Reading:')
shap.summary_plot(shap_values_read, X_train_enc, feature_names=base_feature_names, max_display=15)

print('SHAP Summary — Math:')
shap.summary_plot(shap_values_math, X_train_enc, feature_names=base_feature_names, max_display=15)

## Key Findings

Across both targets the most influential features are consistent:

| Feature | Direction | Note |
|---|---|---|
| `lunch` (free) | Negative | Strongest single predictor; SES proxy |
| `school_id` | Mixed | School-level quality variance |
| `experience` | Positive | More experienced teachers → higher scores |
| `ethnicity` (cauc) | Positive | Reflects broader socioeconomic gaps |
| `class_type` (small) | Positive | Confirms Project STAR class-size findings |
| `gender` (male) | Positive (math) / Negative (reading) | Subject-specific gender gap |